In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.applications import ResNet101
from tensorflow.keras.applications.resnet import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tqdm import tqdm
import matplotlib.pyplot as plt

In [3]:
PROCESSED_IMAGE_DIR = '/kaggle/input/processed-image/processed_images_256'
CSV_FILE_PATH = '/kaggle/input/zenodo2/Zenodo2/Imagewise_Data.csv'
# ---------------------------------------------
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 32
INITIAL_EPOCHS = 10
FINE_TUNE_EPOCHS = 15
LEARNING_RATE = 0.0001

In [4]:
def load_data():
    """Loads and prepares the data, and returns the label encoder."""
    print("Step 1: Loading and preparing data...")
    df = pd.read_csv(CSV_FILE_PATH)
    image_to_label = pd.Series(df['Category'].values, index=df['Image Name']).to_dict()
    
    image_paths = []
    labels = []
    
    processed_images = os.listdir(PROCESSED_IMAGE_DIR)
    
    for img_name in tqdm(processed_images, desc="Matching images to labels"):
        base_name = os.path.splitext(img_name)[0]
        if base_name in image_to_label:
            image_paths.append(os.path.join(PROCESSED_IMAGE_DIR, img_name))
            labels.append(image_to_label[base_name])

    labels_series = pd.Series(labels)
    class_counts = labels_series.value_counts()
    classes_to_remove = class_counts[class_counts < 2].index.tolist()
    if classes_to_remove:
        mask_to_keep = ~labels_series.isin(classes_to_remove)
        image_paths = np.array(image_paths)[mask_to_keep].tolist()
        labels = labels_series[mask_to_keep].tolist()

    label_encoder = LabelEncoder()
    encoded_labels = label_encoder.fit_transform(labels)
    
    print("\nFinal disease classes for training:")
    for i, class_name in enumerate(label_encoder.classes_):
        print(f"  {i}: {class_name}")

    return image_paths, encoded_labels, label_encoder

def create_model(num_classes):
    """Creates the model with ResNet101 and data augmentation."""
    print("\nStep 3: Building the model with ResNet101 and Data Augmentation...")
    
    inputs = Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
    
    x = tf.keras.layers.RandomFlip("horizontal")(inputs)
    x = tf.keras.layers.RandomRotation(0.1)(x)
    x = tf.keras.layers.RandomZoom(0.1)(x)
    x = tf.keras.layers.RandomContrast(0.1)(x)
    
    base_model = ResNet101(include_top=False, weights='imagenet', input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
    base_model.trainable = False
    
    x = base_model(x, training=False)
    
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=predictions)
    
    print("ResNet101 model built successfully.")
    return model

def plot_history(history, initial_epochs):
    """Plots the combined history of initial training and fine-tuning."""
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.figure(figsize=(15, 6))

    plt.subplot(1, 2, 1)
    plt.plot(acc, label='Training Accuracy')
    plt.plot(val_acc, label='Validation Accuracy')
    plt.axvline(x=initial_epochs-1, color='r', linestyle='--', label='Start Fine-Tuning')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')

    plt.subplot(1, 2, 2)
    plt.plot(loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.axvline(x=initial_epochs-1, color='r', linestyle='--', label='Start Fine-Tuning')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')

    plt.tight_layout()
    plt.savefig('fine_tuning_history_resnet101.png')
    plt.show()

In [ ]:
if __name__ == '__main__':
    image_paths, labels, label_encoder = load_data()
    num_classes = len(label_encoder.classes_)
    
    if len(image_paths) == 0:
        print("Error: No data to train on.")
    else:
        #Split Data
        print("\nStep 2: Splitting data...")
        X_train, X_val, y_train, y_val = train_test_split(
            image_paths, labels, test_size=0.2, random_state=42, stratify=labels
        )
        
        #Load images
        print("\nLoading image data into memory...")
        X_train_data = np.array([img_to_array(load_img(path, target_size=IMAGE_SIZE)) for path in tqdm(X_train, desc="Loading train images")])
        X_val_data = np.array([img_to_array(load_img(path, target_size=IMAGE_SIZE)) for path in tqdm(X_val, desc="Loading val images")])

        #Preprocess images for ResNet
        X_train_preprocessed = preprocess_input(X_train_data)
        X_val_preprocessed = preprocess_input(X_val_data)

        #Create and compile the model for the first phase
        model = create_model(num_classes)
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
                      loss='sparse_categorical_crossentropy', 
                      metrics=['accuracy'])
        
        # --- PHASE 1: Train the head ---
        print("\n--- Phase 1: Training the classification head ---")
        history = model.fit(
            X_train_preprocessed, y_train,
            batch_size=BATCH_SIZE,
            epochs=INITIAL_EPOCHS,
            validation_data=(X_val_preprocessed, y_val)
        )

        # --- PHASE 2: Fine-tuning ---
        print("\n--- Phase 2: Fine-tuning the top layers of the model ---")
        base_model = model.get_layer('resnet101')
        base_model.trainable = True
        
        fine_tune_at = 143 
        for layer in base_model.layers[:fine_tune_at]:
            layer.trainable = False

        # Re-compile the model with a very low learning rate
        model.compile(
            loss='sparse_categorical_crossentropy',
            optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
            metrics=['accuracy']
        )
        model.summary()

        # Continue training
        total_epochs = INITIAL_EPOCHS + FINE_TUNE_EPOCHS
        history_fine = model.fit(
            X_train_preprocessed, y_train,
            epochs=total_epochs,
            initial_epoch=history.epoch[-1],
            batch_size=BATCH_SIZE,
            validation_data=(X_val_preprocessed, y_val)
        )
        
        # Append histories for plotting
        history.history['accuracy'].extend(history_fine.history['accuracy'])
        history.history['val_accuracy'].extend(history_fine.history['val_accuracy'])
        history.history['loss'].extend(history_fine.history['loss'])
        history.history['val_loss'].extend(history_fine.history['val_loss'])

        # --- Final Evaluation ---
        print("\n--- Detailed Classification Report after Fine-Tuning ---")
        y_pred_proba = model.predict(X_val_preprocessed)
        y_pred = np.argmax(y_pred_proba, axis=1)
        
        print(classification_report(y_val, y_pred, target_names=label_encoder.classes_))
        
        plot_history(history, INITIAL_EPOCHS)
        
        loss, accuracy = model.evaluate(X_val_preprocessed, y_val, verbose=0)
        print(f"\nFinal Validation Accuracy after Fine-Tuning: {accuracy*100:.2f}%")
        
        model.save('oral_disease_resnet101_finetuned.h5')
        print("\nModel saved to oral_disease_resnet101_finetuned.h5")